# 05 Continuity, the Squeeze Theorem, and the Intermediate Value Theorem

| | |
|---|---|
| **Course** | Calculus I |
| **Prerequisite** | 04 Limits |

A function is continuous at $a$ when three things are equal:
the value $f(a)$, the left-hand limit, and the right-hand limit.
A limit can exist when the value does not. The discontinuity is then removable.
Fill the hole by redefining a single point, and the new function is continuous.
The old function remains undefined there.

## Two theorems

**Squeeze theorem.** If $g(x)\le f(x)\le h(x)$ near $a$, and the limits of $g$ and of $h$ are both $L$, then the limit of $f$ is $L$ as well.

**Intermediate value theorem.** If $f$ is continuous on $[a,b]$ and $k$ lies between $f(a)$ and $f(b)$,
then at least one $c$ in $(a,b)$ satisfies $f(c)=k$.
The theorem shows that a root exists. It does not show where the root sits.


In [1]:
import sympy as sp

# Compare a computed SymPy result with the answer found by hand.
def check(found, hand, label="Result"):
    # Simplify each side first so equivalent algebraic forms can match.
    found_s = sp.simplify(found)
    hand_s = sp.simplify(hand)
    difference = sp.simplify(found_s - hand_s)
    print(label)
    print("  Hand:", hand_s)
    print("  Python:", found_s)
    print("  Difference:", difference)
    # The methods agree only when that difference simplifies to zero.
    assert difference == 0
    print("  Confirmed")


## Example 1 — Three kinds of failure

### Hand solution

$$
f(x)=\dfrac{x^2-1}{x-1}.
$$

The function is undefined at $x=1$, but the limit is $2$. The discontinuity is removable.

$$
g(x)=\dfrac{1}{x}.
$$

The left-hand and right-hand limits at $0$ are infinities of opposite sign. The discontinuity is infinite.

$$
h(x)=\begin{cases}
0 & x<0 \\
1 & x\ge 0
\end{cases}
$$

The left-hand limit is $0$ and the right-hand limit is $1$. The discontinuity is a jump. The value $h(0)=1$ saves only the right-hand side.

### Python recheck


In [2]:
import sympy as sp

# Three failures: a removable hole for (x^2 - 1)/(x - 1),
# opposite infinite limits for 1/x at 0, and a jump in the step function h.
x = sp.symbols("x")
f = (x**2 - 1)/(x - 1)
# Hand limit of f at x = 1 is 2.
assert sp.limit(f, x, 1) == 2
left = sp.limit(1/x, x, 0, dir="-")
right = sp.limit(1/x, x, 0, dir="+")
print("1/x from the left and the right:", left, right)
# Left-hand limit -infinity, right-hand limit +infinity.
assert left == -sp.oo and right == sp.oo
def h(value):
    return 0 if value < 0 else 1

# Sample the jump: negative inputs give 0, positive inputs give 1.
left_samples = [h(-10**(-k)) for k in (1, 2, 3)]
right_samples = [h(10**(-k)) for k in (1, 2, 3)]
print("Left samples:", left_samples)
print("Right samples:", right_samples)
print("h(0) =", h(0))
assert left_samples == [0, 0, 0]
assert right_samples == [1, 1, 1]
# The defined value h(0) = 1 matches only the right-hand side.
assert h(0) == 1
print("Jump confirmed")


1/x from the left and the right: -oo oo
Left samples: [0, 0, 0]
Right samples: [1, 1, 1]
h(0) = 1
Jump confirmed


## Example 2 — Filling the hole

### Hand solution

Define a new function:

$$
F(x)=\begin{cases}
\dfrac{x^2-1}{x-1} & x\neq 1 \\
2 & x=1
\end{cases}
$$

For $x\neq 1$, we have $F(x)=x+1$, and $F(1)=2$ is the same number as the limit.
So $F$ agrees with $x+1$ everywhere, and $F$ is continuous.
The functions $f$ and $F$ are two different functions.

### Python recheck


In [3]:
import sympy as sp

# F fills the hole: F(1) = 2, and elsewhere F uses (x^2 - 1)/(x - 1).
# Hand check: the limit is 2, and F(4) = 5 because the simplified rule is x + 1.
x = sp.symbols("x")
def F(value):
    if value == 1:
        return 2
    return (value**2 - 1)/(value - 1)

print("F(1) =", F(1))
print("Limit =", sp.limit((x**2 - 1)/(x - 1), x, 1))
print("F(4) =", F(4))
assert F(1) == 2
assert sp.limit((x**2 - 1)/(x - 1), x, 1) == 2
assert F(4) == 5
print("Confirmed")


F(1) = 2
Limit = 2
F(4) = 5.0
Confirmed


## Example 3 — The intermediate value theorem, existence only

### Hand solution

$$
f(x)=x^3+x-1.
$$

The function is continuous on $[0,1]$. Also $f(0)=-1<0$ and $f(1)=1>0$.
Zero lies between these two values, so a root exists in $(0,1)$.
The theorem does not give the coordinate of that root.

### Python recheck


In [4]:
import sympy as sp

# f(x) = x^3 + x - 1 changes sign on [0, 1], so a root exists in (0, 1).
x = sp.symbols("x")
f = x**3 + x - 1
print("f(0) =", f.subs(x, 0), " f(1) =", f.subs(x, 1))
# A numerical root near the starting guess 1/2; the theorem does not name it.
root = sp.nsolve(f, x, sp.Rational(1, 2))
print("One numerical root:", root)
assert f.subs(x, 0) < 0 < f.subs(x, 1)
assert 0 < root < 1
print("Confirmed")


f(0) = -1  f(1) = 1
One numerical root: 0.682327803828019
Confirmed


## Example 4 — The squeeze theorem

### Hand solution

$$
-x^2\le x^2\sin\dfrac{1}{x}\le x^2,\qquad x\neq 0,
$$

because $-1\le\sin(1/x)\le 1$.
The two walls both have limit $0$ at $0$, so

$$
\lim_{x\to 0}x^2\sin\dfrac{1}{x}=0.
$$

The function $\sin(1/x)$ itself has no limit. The factor $x^2$ smothers it.
At $x=0$ the original function is undefined. The limit exists.

### Python recheck


In [5]:
import sympy as sp

# x^2 * sin(1/x) is squeezed to 0 at x = 0. Hand limit: 0.
x = sp.symbols("x")
f = x**2 * sp.sin(1/x)
exact = sp.limit(f, x, 0)
print("Limit:", exact)
assert exact == 0
print("Confirmed")


Limit: 0
Confirmed


## Example 5 — Continuity of a piecewise function at the joint

### Hand solution

$$
p(x)=\begin{cases}
\dfrac{\sin x}{x} & x\neq 0 \\
1 & x=0
\end{cases}
$$

The standard limit of $\dfrac{\sin x}{x}$ equals $1$, and the value we defined is $1$ as well.
So $p$ is continuous at $0$. If another number stood in place of $1$, the limit would still be $1$, and continuity would fail.

### Python recheck


In [6]:
import sympy as sp

# The standard limit sin(x)/x at 0 equals 1, which matches the value defined at 0.
x = sp.symbols("x")
assert sp.limit(sp.sin(x)/x, x, 0) == 1
print("Limit of sin(x)/x =", sp.limit(sp.sin(x)/x, x, 0))
print("Defined value at zero = 1")
print("Confirmed")


Limit of sin(x)/x = 1
Defined value at zero = 1
Confirmed


## A pitfall

Continuity is a property of a point, not a property of a formula.
One formula can be continuous everywhere except at a single point.
The intermediate value theorem promises nothing on an open interval, and nothing for a discontinuous function.

## Summary

Compare three numbers at once: the left-hand limit, the right-hand limit, and the value.
Removable means the limit exists and the value does not, or the value is the wrong number.
The squeeze theorem is the tool when an expression is trapped between two functions whose limits are known.
